# Fucrimodo As Library

Fucrimodo can be used as an independent program, or it can be integrated
in form of a python library. This notebook shows how a basic Multi-Stage
GA can be defined.

Since multiple aspects of the algorithm depend on pseudo random numbers,
a seed is set to make everything reproducable. Note: Please always reset
the seed before a rerun of the script, else results are not
reproducable!

In [ ]:
import random
import numpy as np

random.seed(42)
np.random.seed(42)


To store and analyse results a temporary dir is used. You can also
change this to any other dir.

In [ ]:
import tempfile
import os
tmp_dir = tempfile.mkdtemp(prefix="fucrimodo_example_")
print(tmp_dir)


## Target Files Definition

Target files are used as the main input of a fucrimodo run. They store
the target soap descriptor as well as the descriptor type and its
parameters needed to reproduce it.

### Generate file

In a real application of fucrimodo the target structure is unknown.
However, for testing we can create a target file from a known compound.

In [ ]:
import ase

# For testing we generate an simple example stucture
target_structure = ase.Atoms(
     numbers=[26],
     positions=[[0, 0, 0]],
     cell = [
         [3, 0, 0],
         [0, 3, 0],
         [0, 0, 3],
     ],
     pbc=True,
)


To get the target file we set parameters for the descriptor and
calculate the target descriptor features.

In [ ]:
from fucrimodo.core.utils import CustomSOAP
from fucrimodo.utils import target_file_parser

kwargs = {
    "r_cut": 15.0,
    "n_max": 8,
    "l_max": 8,
    "sigma": 0.5,
    "species": list(target_structure.get_chemical_symbols()),
    "periodic": True,
    "average": "inner"
}
soap = CustomSOAP(**kwargs)
target_features = soap.create(target_structure)

target_file_path = os.path.join(tmp_dir, "target_file.json")

target_file_parser.save_to_target_file(
    features = list(target_features),
    descriptor_name = "CustomSOAP",
    descriptor_parameters = kwargs,
    save_path = target_file_path,
    additional_notes = "",
)


To show that the original structure is not needed for the inversion it
can be deleted.

In [ ]:
del target_structure


### Load file

From here on we can pretend the target structure never existed. We now
parse the target file without any knowledge about the target structure.

In [ ]:
soap_obj, target_features, _ = target_file_parser.load_target_file(target_file_path)

species = soap_obj.species


## Generate start population

The GA algorithm needs an initial population to start. To get this
population fucrimodo includes a few start population generator. Here we
use a random sampling approach that is guided by knowledge about the
target descriptor.

We first need to set up some hyper parameters:

In [ ]:
import fucrimodo.core.utils as core_utils
from fucrimodo.customs import fitness_functions as ff

# Defines smallest allowed distance between
#  two neighboring atoms based on their coval radii.
closest_distances = core_utils.CustomClosestDistances(
    species=species, ratio_of_covalent_radii=0.7
)

# Min. and max. cell size of generated structures
cell_bound = core_utils.CustomCellBounds(
    {
        "a": [1, 100],
        "b": [1, 100],
        "c": [1, 100],
        "alpha": [10, 170],
        "beta": [10, 170],
        "gamma": [10, 170],
    }
)

# The RBF similarity is the most important metric to analyse how close
# each candidate resembles the target descriptor.
from fucrimodo.utils.soap_similarity import SOAPSimilarity, RBFSimilarity
soap_sim = RBFSimilarity(
    target_features,
    descriptor_object = soap_obj,
    rbf_gamma = 0.01,
)
similarity_fitnesses = ff.SimilarityToTargetSOAPFitness(
    target_features,
    soap_object = soap_obj,
    soap_similarity = soap_sim,
    db_title = "small_rbf_sim_fitness"
)

# Set an estimated number of atoms as hyper param.
# If unknown: Repeat the algorithm with different values.
n_atoms = 1

# Please specify the number of physical cores that can be used for multiprocessing
n_cores = 16


In [ ]:
# Generate a population with estimates based on the SOAP descriptor
from fucrimodo.customs import population_generator as pop_gen
population_generator = pop_gen.RandomSampleCrystalPopulation(
    soap_obj=soap_obj,
    target_features=target_features,
    closest_distances=closest_distances,
    fitness_functions=similarity_fitnesses,
    n_atoms=n_atoms,
    n_jobs=n_cores,
)

population = population_generator.generate_population(100)


The individuals of the start population can now be investigated.

In [ ]:
from ase.visualize import view

view(population.individuals)


## Set up the Multi-Stage Genetic Algorithm

In [ ]:
save_dir = tmp_dir

# Define a global statistic that should be tracked during all runs.
# Here the SOAP similarity of the candidate to the target is used.
rbf_similarity_fitness = ff.SimilarityToTargetSOAPFitness(
    target_soap_features=target_features,
    soap_object=soap_obj,
    soap_similarity=RBFSimilarity(
        target_feature_vector=target_features,
        rbf_gamma=0.1,
        adjust_gamma=False,
    ),
    db_title="rbf",
    round_result=None,
)

global_statistics_dict = {
    "RBF_Similarity": rbf_similarity_fitness.evaluate_individual,
}

from fucrimodo.core.multi_stage_search import MultiStageSearch
multi_stage_search = MultiStageSearch(
    save_dir=save_dir,
    target_features=target_features,
    descriptor_object=soap_obj,
    descriptive_name="notebook_example_run",
    global_statistics_dict=global_statistics_dict,
)


Set a global break condition that should be used during and after each
stage. In this case the first fitness function (set to S$_{0.1}$) should
not excede 0.99.

In [ ]:
from fucrimodo.customs.ga_stage import break_conditions
global_break_condition = break_conditions.MaxFitnessBreak(0, 0.99)


### Define Exploration GA Stage

This stage explores the search space for structure.

First import all necessary libraries:

In [ ]:
from fucrimodo.customs import fitness_functions as ff
from fucrimodo.customs.ga_stage import (
    GAStage,
    crossovers,
    mutations,
)
from fucrimodo.customs.ga_stage.presets import (
    get_soap_similarity_fitness_list,
    get_species_specific_soap_fitness_list,
)
from fucrimodo.customs import population_selections


In [ ]:
name = "explore_ga"

n_generations = 100

description = "This stage explores the search space for structure."

fitness_func_list =  [
    rbf_similarity_fitness,
    ff.PhysicalityFitness(closest_distances),
]

mutation_list = [
    mutations.pos_mut.RattleMutation(
        closest_distances=closest_distances,
        rattle_strength=0.5,
        rattle_prop=0.8,
    ),
    mutations.cell_mut.StrainMutation(
        closest_distances=closest_distances,
        n_variable_cell_vectors=3,
        stddev=0.1,
    ),
    mutations.cell_mut.StrainMutation(
        closest_distances=closest_distances,
        n_variable_cell_vectors=1,
        stddev=0.3,
    ),
    mutations.sym_mut.GetConventionalCellMutation(
        closest_distances=closest_distances, symmetry_tol=0.3
    ),
    mutations.cell_mut.MinimizeTiltMutation(closest_distances),
]
mutation_list.append(
    mutations.multi_mut.MultipleMutations(
        mutation_list, closest_distances, 2
    )
)

crossover_list = [
    crossovers.CutAndSpliceCrossover(
        closest_distances=closest_distances, cell_bounds=cell_bound
    ),
    crossovers.CutAndSpliceCrossover(
        closest_distances=closest_distances,
        cell_bounds=cell_bound,
        number_of_variable_cell_vectors=3,
    ),
    crossovers.UnitCellCrossover(closest_distances=closest_distances),
]

break_condition = break_conditions.MultipleOrBreak(
    [
        break_conditions.GenerationBreak(n_generations),
        global_break_condition,
    ]
)

parent_selection = population_selections.TournamentSelection(5)

survivor_selection = population_selections.NSGA2Selection()

explore_ga_stage = GAStage(
    name=name,
    fitness_functions=fitness_func_list,
    crossover_list=crossover_list,
    mutation_list=mutation_list,
    mutation_probability=0.9,
    crossover_probability=0.9,
    break_condition=break_condition,
    parent_selection=parent_selection,
    survivor_selection=survivor_selection,
    parent_ratio=0.5,
    description=description,
    save_n_crystals=10,
)


### Define Optimization GA Stage

This stage optimizes the found structures.

In [ ]:
name = "optimize_ga"

n_generations = 500

description = "This stage optimizes the found structure."

fitness_func_list =  [
    rbf_similarity_fitness,
]

mutation_list = [
    mutations.pos_mut.RattleMutation(
        closest_distances=closest_distances,
        rattle_strength=0.1,
        rattle_prop=0.8,
    ),
    mutations.pos_mut.RattleMutation(
        closest_distances=closest_distances,
        rattle_strength=0.2,
        rattle_prop=0.8,
    ),
    mutations.cell_mut.StrainMutation(
        closest_distances=closest_distances,
        n_variable_cell_vectors=3,
        stddev=0.1,
    ),
    mutations.cell_mut.StrainMutation(
        closest_distances=closest_distances,
        n_variable_cell_vectors=1,
        stddev=0.2,
    ),
    mutations.sym_mut.GetConventionalCellMutation(
        closest_distances=closest_distances, symmetry_tol=0.3
    ),
    mutations.cell_mut.MinimizeTiltMutation(closest_distances),
]
mutation_list.append(
    mutations.multi_mut.MultipleMutations(
        mutation_list, closest_distances, 2
    )
)

crossover_list = [
    crossovers.CutAndSpliceCrossover(
        closest_distances=closest_distances, cell_bounds=cell_bound
    ),
    crossovers.CutAndSpliceCrossover(
        closest_distances=closest_distances,
        cell_bounds=cell_bound,
        number_of_variable_cell_vectors=3,
    ),
    crossovers.UnitCellCrossover(closest_distances=closest_distances),
]

break_condition = break_conditions.MultipleOrBreak(
    [
        break_conditions.GenerationBreak(n_generations),
        global_break_condition,
    ]
)

parent_selection = population_selections.TournamentSelection(3)

survivor_selection = population_selections.NSGA2Selection()

optimize_ga_stage = GAStage(
    name=name,
    fitness_functions=fitness_func_list,
    crossover_list=crossover_list,
    mutation_list=mutation_list,
    mutation_probability=0.5,
    crossover_probability=0.8,
    break_condition=break_condition,
    parent_selection=parent_selection,
    survivor_selection=survivor_selection,
    parent_ratio=0.5,
    description=description,
    save_n_crystals=10,
)


## Run GA Stages

Run to exploration stage on the initial population:

In [ ]:
multi_stage_search.run(population=population, stage=explore_ga_stage)


The population after the exploration can now be analysed.

In [ ]:
view(population.individuals)

all_fitness_values = [ind.fitness.values[0] for ind in population.individuals]
print("Max fitness:", max(all_fitness_values))


Now the optimization stage can be run:

In [ ]:
multi_stage_search.run(population=population, stage=optimize_ga_stage)


## Analyse Run

The directory of the run can be found at the location

In [ ]:
multi_stage_search.run_dir


### Retrieving Structures

All generated structures are stored in the run directory inside of a
\`ASE\` database called \`crystals.db\`. Structures can be analysed here
or with the cli/web interface provided by \`ASE\` ((more info in their
docs)\[<https://docs.ase-lib.org/ase/db/db.html>\])

In [ ]:
from ase.db import connect

db_path = os.path.join(multi_stage_search.run_dir, "crystals.db")
structure_db = connect(db_path)

# e.g. get first structure of db
first_structure = structure_db.get_atoms(1)
print(first_structure)
view(first_structure)


### Analyse Run Metrics

To make analysis of metrics simple, \`fucrimodo\` comes with some helper
classes to organize the results. The run directory of a finished run can
be loaded into the \`RunData\` class.

In [ ]:
from fucrimodo.analysis.run_analysis import RunData, get_global_statistics_overview
run_data = RunData(multi_stage_search.run_dir)


From this you can get some information.

In [ ]:
print("Total number of generations:", run_data.total_generations)
print("Number of stages performed:", run_data.n_stages)
print()

# You can also get the structures
structure_list = run_data.crystals

# Or use a helper to get an overview of all global metrics (Here only one metric was used)
print("Global Metric Statistics:")
print(get_global_statistics_overview(run_data))


Finally you can also plot the statistics.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots()

results_df = run_data.global_statistics.loc[0, "results"]
results_df.plot(
    ax=ax,
    x="gen",
    y=["min", "max", "avg"],
    linewidth=2.0,
)
ax.set_xlabel("Generation")
ax.set_ylabel("Similarity")

# Plot a line where the stages change
stage_ids = results_df["stage_id"].to_numpy()
change_idx = np.flatnonzero(stage_ids[1:] != stage_ids[:-1])
ax.vlines(change_idx, -0, 1, "black", zorder=0)

ax.set_ylim([0, 1])
ax.set_xlim([0, len(stage_ids)])

